# V1DD CAVE synapse download

Download incoming synapses for dendrite-proofread roots in `v1dd`, materialization **1196**, using the Delta Lake export partitioned by `post_pt_root_id`. Presynaptic roots are not restricted.

Requires `caveclient`, `polars`, and `deltalake` (already declared in this repository), plus locally configured CAVE credentials for the V1DD server. Both Delta readers use anonymous GCS access. Availability of the v1196 export must be confirmed when running.

The full download may be large. Set `root_id_limit = 10` for a trial before downloading the full selection. Output files use the absolute directory `/workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196`, independent of the notebook location or kernel working directory.

In [1]:
import bisect
import time
from pathlib import Path

import polars as pl
from caveclient import CAVEclient
from deltalake import DeltaTable

In [2]:
CAVEclient.setup_token("https://global.em.brain.allentech.org")  # for v1dd

Visit https://global.em.brain.allentech.org/sticky_auth/settings/tokens and copy an existing token or create a new one.
All datastacks are configured to use this server address.
You will not need to specify a server address when initializing a client for configured datastacks in the future.
Setup complete!


In [3]:
version = 1196
datastack = "v1dd"
server_address = "https://global.em.brain.allentech.org"
storage_options = {"GOOGLE_SKIP_SIGNATURE": "true"}
root_id_limit = None

table_path = (
    f"gs://cave-allen-materialization-dump/deltalake_exports/{datastack}/"
    f"v{version}/synapses_v1dd/post_pt_root_id"
)
output_suffix = "" if root_id_limit is None else f"_first_{root_id_limit}_roots"
output_path = (
    Path("/workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196")
    / f"v1dd_v{version}_proofread_dendrite_synapses{output_suffix}.parquet"
)
print(f"Delta export: {table_path}")
print(f"Output: {output_path.resolve()}")

Delta export: gs://cave-allen-materialization-dump/deltalake_exports/v1dd/v1196/synapses_v1dd/post_pt_root_id
Output: /workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196/v1dd_v1196_proofread_dendrite_synapses.parquet


## Select proofread dendrites

The export is partitioned by `post_pt_root_id`, so the root selection below is postsynaptic and dendrite proofreading is the matching status. Use the same materialization version for the proofreading query and the synapse export. The optional trial selects a reproducible subset of sorted root IDs.

In [4]:
client = CAVEclient(datastack, server_address=server_address, version=version)
# status_dendrite arrives as 't'/'f' strings when the client streams results via CSV, so filter server-side.
proofread_table = client.materialize.query_table(
    "proofreading_status_and_strategy",
    filter_equal_dict={"status_dendrite": True},
)
proofread_root_ids = sorted(proofread_table["pt_root_id"].dropna().unique().tolist())
print(f"Number of proofread root IDs: {len(proofread_root_ids):,}")

if root_id_limit is not None:
    if not isinstance(root_id_limit, int) or root_id_limit <= 0:
        raise ValueError("root_id_limit must be None or a positive integer.")
    proofread_root_ids = proofread_root_ids[:root_id_limit]
if not proofread_root_ids:
    raise ValueError("No proofread dendrite root IDs were selected.")
print(f"Root IDs selected for download: {len(proofread_root_ids):,}")

Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


Number of proofread root IDs: 3,249
Root IDs selected for download: 3,249


## Build the filtered scan

Sorted file maxima are assumed to correspond to consecutive, zero-based `post_pt_root_id_partition` values in this export. The root-ID filter is applied as well. Drop the position, supervoxel, validity, and partition columns, retaining the remaining synapse attributes.

In [5]:
delta_table = DeltaTable(table_path, storage_options=storage_options)
add_actions = pl.from_arrow(delta_table.get_add_actions(flatten=True))
boundaries = add_actions.sort("min.post_pt_root_id")["max.post_pt_root_id"].to_list()
proofread_partition_ids = sorted(
    {bisect.bisect_left(boundaries, root_id) for root_id in proofread_root_ids}
)
print(f"Candidate partitions: {len(proofread_partition_ids):,}")

table_plan = pl.scan_delta(table_path, storage_options=storage_options)
table_plan = table_plan.filter(
    pl.col("post_pt_root_id_partition").is_in(proofread_partition_ids)
    & pl.col("post_pt_root_id").is_in(proofread_root_ids)
)
table_plan = table_plan.drop(
    "valid",
    "post_pt_position",
    "pre_pt_position",
    "pre_pt_supervoxel_id",
    "post_pt_supervoxel_id",
    "post_pt_root_id_partition",
)

DeltaError: Kernel error: Error interacting with object store: The operation lacked the necessary privileges to complete for path deltalake_exports/v1dd/v1196/synapses_v1dd/post_pt_root_id/_delta_log/_last_checkpoint: Error performing GET https://storage.googleapis.com/cave%2Dallen%2Dmaterialization%2Ddump/deltalake%5Fexports%2Fv1dd%2Fv1196%2Fsynapses%5Fv1dd%2Fpost%5Fpt%5Froot%5Fid%2F%5Fdelta%5Flog%2F%5Flast%5Fcheckpoint in 213.919833ms - Server returned non-2xx status code: 403 Forbidden: <?xml version='1.0' encoding='UTF-8'?><Error><Code>AccessDenied</Code><Message>Access denied.</Message><Details>Anonymous caller does not have storage.objects.get access to the Google Cloud Storage object. Permission 'storage.objects.get' denied on resource '//storage.googleapis.com/projects/_/buckets/cave-allen-materialization-dump/objects/deltalake_exports/v1dd/v1196/synapses_v1dd/post_pt_root_id/_delta_log/_last_checkpoint' (or it may not exist).</Details></Error>

## Download to Parquet

This cell streams the filtered data to disk. It refuses to overwrite an existing file; choose a new `output_path` or deliberately remove the previous output before rerunning.

In [ ]:
if output_path.exists():
    raise FileExistsError(f"Output already exists: {output_path.resolve()}")

start = time.perf_counter()
table_plan.sink_parquet(output_path, engine="streaming")
elapsed = time.perf_counter() - start
print(f"Time taken to write filtered table: {elapsed:.1f} seconds")
print(f"Saved: {output_path.resolve()}")

## Inspect the download

Count rows and display a small preview without loading the entire file into memory.

In [ ]:
table = pl.scan_parquet(output_path)
row_count = table.select(pl.len()).collect().item()
print(f"Number of rows in filtered table: {row_count:,}")
table.head(10).collect()